In [ ]:
import pandas as pd
from pathlib import Path
import zipfile
import shutil
import sys
import subprocess
import os

In [ ]:

input_root = Path('/kaggle/input')
working_root = Path('/kaggle/working')
project_roots = [p.parent for p in input_root.rglob('pyproject.toml') if (p.parent / 'src' / 'cosmic_black_swan').is_dir()]
if not project_roots:
    archives = list(input_root.rglob('*.zip'))
    if len(archives) != 1:
        raise FileNotFoundError(f'Expected one project ZIP under /kaggle/input; found: {archives}')
    project_home = working_root / 'cbs-project'
    project_home.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or '..' in member_path.parts:
                raise ValueError(f'Unsafe path in project ZIP: {member.filename}')
        archive.extractall(project_home)
    project_roots = [p.parent for p in project_home.rglob('pyproject.toml') if (p.parent / 'src' / 'cosmic_black_swan').is_dir()]
if len(project_roots) != 1:
    raise RuntimeError(f'Expected exactly one Cosmic Black Swan project root; found {project_roots}')
PROJECT = project_roots[0]
DATA_ROOT = working_root / 'cbs-data'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print('Project:', PROJECT)
print('Data:', DATA_ROOT)

In [ ]:

source_project = PROJECT
PROJECT = Path('/kaggle/working/cbs-project-src')
if source_project.resolve() != PROJECT.resolve():
    shutil.copytree(source_project, PROJECT, dirs_exist_ok=True)

print('Project copy:', PROJECT)

In [ ]:

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--verbose", "-e", str(PROJECT)],
    capture_output=True,
    text=True
)

print("Exit code:", result.returncode)
print(result.stdout[-6000:])
print(result.stderr[-6000:])

In [ ]:

def run(args):
    args = [str(x) for x in args]
    print('$', ' '.join(args))
    subprocess.run(args, check=True)

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for secret_name in ['GOOGLE_DRIVE_OAUTH_TOKEN_JSON', 'GEMINI_API_KEY', 'GOOGLE_DRIVE_FOLDER_ID']:
        try:
            os.environ[secret_name] = secrets.get_secret(secret_name)
            print(f'Loaded Kaggle Secret: {secret_name}')
        except Exception:
            continue
except Exception as exc:
    print('Kaggle Secrets are unavailable; pipeline work can continue locally in this session.', type(exc).__name__)

RESTORE_SNAPSHOT_ID = ''
if RESTORE_SNAPSHOT_ID:
    run(['cbs', 'drive-restore', '--snapshot-id', RESTORE_SNAPSHOT_ID, '--destination', DATA_ROOT])

In [ ]:
from astroquery.mast import Observations, conf

with conf.set_temp("timeout", 90):
    page = Observations.query_criteria(
        obs_collection="TESS",
        dataproduct_type="timeseries",
        sequence_number=1,
        pagesize=100,
        page=1,
    )

print("Observation rows returned:", len(page))
print(page[:5])

In [ ]:
import socket

try:
    print("MAST DNS resolved to:", socket.gethostbyname("mast.stsci.edu"))
except Exception as e:
    print("MAST DNS failed:", type(e).__name__, e)

In [ ]:
products = Observations.get_product_list(page)
print("Product rows:", len(products))
print("Columns:", products.colnames)

if "productFilename" in products.colnames:
    names = [str(x) for x in products["productFilename"]]
    light_curves = [x for x in names if x.lower().endswith("lc.fits")]
    print("Light-curve FITS products:", len(light_curves))
    print(*light_curves[:5], sep="\n")

run(["cbs", "fetch", "--data-root", DATA_ROOT,
     "--sectors", "1-10", "--max-lightcurves", "5000",
     "--per-sector-cap", "500"])

In [ ]:
from pathlib import Path

fetch_file = Path("/kaggle/working/cbs-project-src/src/cosmic_black_swan/fetch.py")
source = fetch_file.read_text(encoding="utf-8")

marker = "            sequence_number=int(sector),"
page_args = "            pagesize=100,\n            page=1,"

if page_args not in source:
    if source.count(marker) != 1:
        raise RuntimeError("Could not safely locate the MAST query; stop here.")
    source = source.replace(marker, marker + "\n" + page_args, 1)

source = source.replace("                verbose=False,", "                verbose=True,", 1)
fetch_file.write_text(source, encoding="utf-8")
print("Fetch limited to the first 100 observations per sector.")

run([
    "cbs", "fetch",
    "--data-root", DATA_ROOT,
    "--sectors", "1",
    "--max-lightcurves", "10",
    "--per-sector-cap", "10",
    "--batch-size", "5",
])

In [ ]:
run(["cbs", "featurize", "--data-root", DATA_ROOT])

In [ ]:
run(["cbs", "score", "--data-root", DATA_ROOT, "--contamination", "0.02"])

In [ ]:
run([
    "cbs", "fetch",
    "--data-root", DATA_ROOT,
    "--sectors", "1",
    "--max-lightcurves", "30",
    "--per-sector-cap", "30",
    "--batch-size", "5",
])

In [ ]:
run(["cbs", "featurize", "--data-root", DATA_ROOT, "--append"])

In [ ]:
run(["cbs", "score", "--data-root", DATA_ROOT, "--contamination", "0.02"])

In [ ]:
display(pd.read_csv(DATA_ROOT / "scores" / "isolation_forest.csv").head(10))

In [ ]:
from pathlib import Path
import shutil
import pandas as pd

DATA_ROOT = Path("/kaggle/working/cbs-data")

if not (DATA_ROOT / "manifest.csv").exists():
    matches = [
        p for p in Path("/kaggle/input").rglob("manifest.csv")
        if p.parent.name == "cbs-data"
    ]
    preferred = [
        p for p in matches
        if "cosmic-swan-phase-3" in str(p).lower()
    ]

    if preferred:
        saved_manifest = preferred[0]
    elif len(matches) == 1:
        saved_manifest = matches[0]
    else:
        print("Found manifests:", matches)
        raise FileNotFoundError(
            "Attach the saved Cosmic Swan Phase 3 notebook under Input > Notebooks, "
            "then rerun this cell."
        )

    shutil.copytree(saved_manifest.parent, DATA_ROOT, dirs_exist_ok=True)
    print("Restored data from:", saved_manifest.parent)

manifest = pd.read_csv(DATA_ROOT / "manifest.csv")
print("Manifest rows:", len(manifest))
print("Statuses:", manifest["status"].value_counts().to_dict())
print("FITS files:", len(list((DATA_ROOT / "raw").rglob("*.fits"))))

In [ ]:
PROJECT = Path("/kaggle/working/cbs-project-src")
fetch_file = PROJECT / "src" / "cosmic_black_swan" / "fetch.py"

source = fetch_file.read_text(encoding="utf-8")

if "CBS_PAGINATED_FETCH_V1" in source:
    print("Pagination patch is already present.")
else:
    start_marker = "        observations = Observations.query_criteria("
    end_marker = '        if "productFilename" not in frame or "dataURI" not in frame:'

    start = source.index(start_marker)
    end = source.index(end_marker, start)

    replacement = '''        # CBS_PAGINATED_FETCH_V1: inspect at most 3 pages of 100 observations.
        product_pages = []
        page_size = 100
        max_pages = 3

        for page_number in range(1, max_pages + 1):
            observations = Observations.query_criteria(
                obs_collection="TESS",
                dataproduct_type="timeseries",
                sequence_number=int(sector),
                pagesize=page_size,
                page=page_number,
            )
            if observations is None or len(observations) == 0:
                break

            page_products = Observations.get_product_list(observations)
            if page_products is not None and len(page_products) > 0:
                product_pages.append(page_products)

            if len(observations) < page_size:
                break
            if pause_seconds:
                time.sleep(pause_seconds)

        if not product_pages:
            continue

        if len(product_pages) == 1:
            products = product_pages[0]
        else:
            from astropy.table import vstack
            products = vstack(product_pages, metadata_conflicts="silent")

        frame = _mast_table_frame(products)
'''

    source = source[:start] + replacement + source[end:]
    fetch_file.write_text(source, encoding="utf-8")
    print("Pagination patch applied:", fetch_file)

In [ ]:
import sys
import subprocess

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps", "-e", str(PROJECT),
    ],
    check=True,
)

In [ ]:
run([
    "cbs", "fetch",
    "--data-root", DATA_ROOT,
    "--sectors", "1",
    "--max-lightcurves", "100",
    "--per-sector-cap", "100",
    "--batch-size", "5",
])

In [ ]:
run(["cbs", "featurize", "--data-root", DATA_ROOT, "--append"])

In [ ]:
run(["cbs", "score", "--data-root", DATA_ROOT, "--contamination", "0.02"])